# University Management System
## DBS Project Part 2

| **Name** | 
|---|
| Ali Zafar Awan |
| M Hassan Malik |
| Rabbaya Mughees |
| Fizza Atif |

# **Task 01: Pipeline Setup and Database Integration**

In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-classic transformers accelerate bitsandbytes
print("Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires r

In [2]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
import os
import re
import sqlite3

import pandas as pd
import torch

from IPython.display import display

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_community.utilities.sql_database import SQLDatabase
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import create_sql_query_chain

In [3]:
from kaggle_secrets import UserSecretsClient

client = UserSecretsClient()
token = client.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = token

print("Hugging Face access ready ✅")

Hugging Face access ready ✅


In [4]:
CUSTOM_TABLE_INFO = {
    "ADDRESS": (
        "Contains mailing address details for students and faculty members. "
        "Primary key: address_id. Includes house_no, street_lane, sector_block_phase, "
        "area_town_mohalla, city, province, country, and postal_code."
    ),

    "DEPARTMENT": (
        "University departments such as Computer Science or Business Administration. "
        "Primary key: dept_id. dept_name must be unique. "
        "hod_id links to FACULTY.faculty_id and can be empty."
    ),

    "FACULTY": (
        "Stores teaching and administrative staff records. "
        "Primary key: faculty_id. Includes first_name, last_name, job_title, dob, gender, "
        "phone, email, address_id, and dept_id. "
        "Foreign keys: dept_id -> DEPARTMENT, address_id -> ADDRESS."
    ),

    "DEGREE": (
        "Degree programs offered by the university like BSCS or MBA. "
        "Primary key: degree_id. Includes degree_title, duration in years, and dept_id. "
        "Foreign key: dept_id -> DEPARTMENT."
    ),

    "SEMESTER": (
        "Tracks academic semesters and their timelines. "
        "Primary key: sem_id. Includes sem_session, start_date, end_date, and sem_status."
    ),

    "COURSE": (
        "University courses such as Data Structures or Calculus. "
        "Primary key: course_id. Includes course_name and credit_hours. "
        "Foreign key: dept_id -> DEPARTMENT."
    ),

    "SECTION": (
        "Represents a course section offered in a semester by a faculty member. "
        "Primary key: section_id. Includes section_name, capacity, faculty_id, course_id, and sem_id. "
        "Foreign keys: faculty_id -> FACULTY, course_id -> COURSE, sem_id -> SEMESTER."
    ),

    "STUDENT": (
        "Stores registered student records. "
        "Primary key: student_id formatted like STU000001. "
        "Includes first_name, last_name, batch, dob, gender, phone, email, address_id, and degree_id. "
        "Foreign keys: degree_id -> DEGREE, address_id -> ADDRESS."
    ),

    "ENROLLMENT": (
        "Connects students with the sections they are enrolled in. "
        "Primary key: enrollment_id. Includes enrollment_date, final_grade, student_id, and section_id. "
        "Foreign keys: student_id -> STUDENT, section_id -> SECTION. "
        "CRITICAL: To connect ENROLLMENT back to a COURSE, you must join through the SECTION table using section_id."
    ),

    "ATTENDANCE": (
        "Maintains daily attendance records for enrolled students. "
        "Primary key: attendance_id. Includes date, status, and enrollment_id. "
        "Values for status column are single characters: 'P' for Present, 'A' for Absent, 'L' for Late. "
        "Foreign key: enrollment_id -> ENROLLMENT. "
        "CRITICAL: To connect ATTENDANCE back to a STUDENT, you must join through ENROLLMENT using enrollment_id."
    ),

    "FEES": (
        "Stores course fee information for each enrollment. "
        "Primary key: fee_id. Includes amount, per_credit_rate, and enrollment_id. "
        "Foreign key: enrollment_id -> ENROLLMENT."
    ),

    "FEE_PAYMENT": (
        "Tracks semester-wise fee payments for students. "
        "Primary key: payment_id. Includes total_amount_due, amount_paid, payment_date, "
        "payment_method, payment_status, student_id, and sem_id. "
        "Foreign keys: student_id -> STUDENT, sem_id -> SEMESTER."
    ),
}

print(f"{len(CUSTOM_TABLE_INFO)} tables loaded successfully ✅")

12 tables loaded successfully ✅


In [5]:
# Known schema: tables and their columns
SCHEMA_TABLES = {
    "ADDRESS":      {"address_id", "house_no", "street_lane", "sector_block_phase",
                     "area_town_mohalla", "city", "province", "country", "postal_code"},
    "DEPARTMENT":   {"dept_id", "dept_name", "hod_id"},
    "FACULTY":      {"faculty_id", "first_name", "last_name", "job_title", "dob",
                     "gender", "phone", "email", "address_id", "dept_id"},
    "DEGREE":       {"degree_id", "degree_title", "duration", "dept_id"},
    "SEMESTER":     {"sem_id", "sem_session", "start_date", "end_date", "sem_status"},
    "COURSE":       {"course_id", "course_name", "credit_hours", "dept_id"},
    "SECTION":      {"section_id", "section_name", "capacity", "faculty_id",
                     "course_id", "sem_id"},
    "STUDENT":      {"student_id", "first_name", "last_name", "batch", "dob",
                     "gender", "phone", "email", "address_id", "degree_id"},
    "ENROLLMENT":   {"enrollment_id", "enrollment_date", "final_grade",
                     "student_id", "section_id"},
    "ATTENDANCE":   {"attendance_id", "date", "status", "enrollment_id"},
    "FEES":         {"fee_id", "amount", "per_credit_rate", "enrollment_id"},
    "FEE_PAYMENT":  {"payment_id", "total_amount_due", "amount_paid", "payment_date",
                     "payment_method", "payment_status", "student_id", "sem_id"},
}

def check_out_of_scope(question):
    """
    Scans the question for capitalised identifiers not present in the schema.
    Returns a list of unrecognised tokens (empty = everything looks in-scope).
    """
    import re
    known_tables  = {t.lower() for t in SCHEMA_TABLES}
    known_columns = {c.lower() for cols in SCHEMA_TABLES.values() for c in cols}
    known_tokens  = known_tables | known_columns

    candidates = re.findall(r'\b([A-Z][A-Z_]{2,}|[A-Z][a-z]+[A-Z][a-zA-Z]*)\b', question)
    unrecognised = [w for w in candidates if w.lower() not in known_tokens]
    return list(set(unrecognised))

print(f"Schema validation helper loaded — {len(SCHEMA_TABLES)} tables registered ✅")

Schema validation helper loaded — 12 tables registered ✅


In [6]:
DB_PATH = "/kaggle/input/datasets/alizafarawan/uni-dbs-group1/uni_management.db"

db = SQLDatabase.from_uri(
    f"sqlite:///{DB_PATH}",
    custom_table_info=CUSTOM_TABLE_INFO,
    sample_rows_in_table_info=3
)

print(db.get_context())

{'table_info': "Connects students with the sections they are enrolled in. Primary key: enrollment_id. Includes enrollment_date, final_grade, student_id, and section_id. Foreign keys: student_id -> STUDENT, section_id -> SECTION. CRITICAL: To connect ENROLLMENT back to a COURSE, you must join through the SECTION table using section_id.\n\nContains mailing address details for students and faculty members. Primary key: address_id. Includes house_no, street_lane, sector_block_phase, area_town_mohalla, city, province, country, and postal_code.\n\nDegree programs offered by the university like BSCS or MBA. Primary key: degree_id. Includes degree_title, duration in years, and dept_id. Foreign key: dept_id -> DEPARTMENT.\n\nMaintains daily attendance records for enrolled students. Primary key: attendance_id. Includes date, status, and enrollment_id. Values for status column are single characters: 'P' for Present, 'A' for Absent, 'L' for Late. Foreign key: enrollment_id -> ENROLLMENT. CRITICAL:

In [7]:
conn = sqlite3.connect(DB_PATH)

summary = pd.read_sql_query("""
SELECT 'STUDENT' AS table_name, COUNT(*) AS rows FROM STUDENT
UNION ALL
SELECT 'FACULTY', COUNT(*) FROM FACULTY
UNION ALL
SELECT 'DEPARTMENT', COUNT(*) FROM DEPARTMENT
UNION ALL
SELECT 'COURSE', COUNT(*) FROM COURSE
UNION ALL
SELECT 'SECTION', COUNT(*) FROM SECTION
UNION ALL
SELECT 'ENROLLMENT', COUNT(*) FROM ENROLLMENT
UNION ALL
SELECT 'ATTENDANCE', COUNT(*) FROM ATTENDANCE
UNION ALL
SELECT 'FEES', COUNT(*) FROM FEES
UNION ALL
SELECT 'FEE_PAYMENT', COUNT(*) FROM FEE_PAYMENT
""", conn)

conn.close()

display(summary)

,table_name,rows
0,STUDENT,500
1,FACULTY,48
2,DEPARTMENT,8
3,COURSE,30
4,SECTION,120
5,ENROLLMENT,1251
6,ATTENDANCE,3767
7,FEES,1251
8,FEE_PAYMENT,500


In [9]:
hf_token = UserSecretsClient().get_secret("HF_TOKEN")

MODEL_ID = "defog/sqlcoder-7b-2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    token=hf_token,
)

hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
)

llm   = HuggingFacePipeline(pipeline=hf_pipe)
chain = create_sql_query_chain(llm=llm, db=db)

print(f"model on: {next(model.parameters()).device}")
print("chain ready")

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


model on: cuda:0
chain ready


In [10]:
def ask(question):
    print(f"\nQuestion : {question}")
    print("-" * 60)

    # ── 1. Out-of-scope check ──────────────────────────────────────────────────
    unknown = check_out_of_scope(question)
    if unknown:
        print(f"[Out-of-scope] The following term(s) were not found in the "
              f"database schema and cannot be queried: {unknown}\n"
              f"Available tables: {list(SCHEMA_TABLES.keys())}")
        return None

    # ── 2. Generate SQL via LangChain chain ────────────────────────────────────
    try:
        raw = chain.invoke({"question": question})
    except Exception as e:
        print(f"[chain error] {e}")
        return None

    sql = None
    for pat in [
        r"SQLQuery:\s*(SELECT[\s\S]+?)(?:SQLResult|Answer|$)",
        r"(SELECT[\s\S]+?);",
        r"(SELECT[\s\S]+)",
    ]:
        m = re.search(pat, raw, re.IGNORECASE)
        if m:
            sql = m.group(1).strip().rstrip(";") + ";"
            break

    if not sql:
        print(f"[no SQL extracted]\nRaw output:\n{raw}")
        return None

    sql = sql.replace(" ilike ", " LIKE ").replace(" ILIKE ", " LIKE ")
    print(f"SQL       : {sql}")

    # ── 3. Execute ─────────────────────────────────────────────────────────────
    try:
        conn_tmp = sqlite3.connect(DB_PATH)
        df = pd.read_sql_query(sql, conn_tmp)
        conn_tmp.close()
    except Exception as e:
        print(f"[exec error] {e}")
        return None

    print(f"Result    : {len(df)} row(s)")
    display(df)

    # ── 4. Natural-language summary ────────────────────────────────────────────
    # KEY FIX: We call hf_pipe (the raw HuggingFace pipeline) DIRECTLY here,
    # NOT chain.invoke(). The chain always routes through SQLcoder which only
    # generates SQL. hf_pipe can produce plain English when prompted clearly.
    if len(df) == 0:
        print("\n📋 Summary: No records matched the query.")
    else:
        rows_preview = df.head(5).to_dict(orient="records")
        summary_prompt = (
            f"Question: {question}\n"
            f"Results: {len(df)} rows returned. Columns: {list(df.columns)}.\n"
            f"Sample data: {rows_preview}\n\n"
            "Task: Write a 2-3 sentence plain English summary answering the question "
            "using specific names and numbers from the data. No SQL. No code. Just text."
        )
        try:
            # Call hf_pipe directly — bypasses the SQL chain entirely
            raw_summary = hf_pipe(
                summary_prompt,
                max_new_tokens=150,
                do_sample=False,
                return_full_text=False   # only return new tokens, not the prompt echoed back
            )
            summary_text = raw_summary[0]["generated_text"].strip()

            # Strip any SQL that leaked into the output (SQLcoder habit)
            summary_text = re.sub(
                r"(SELECT|FROM|WHERE|JOIN|GROUP BY|ORDER BY|INSERT|UPDATE|DELETE)[\s\S]*",
                "", summary_text, flags=re.IGNORECASE
            ).strip()

            # If the model produced nothing useful, fall back to a rule-based summary
            if len(summary_text) < 15:
                first_col = df.columns[0]
                first_val = df.iloc[0, 0]
                count = len(df)
                summary_text = (
                    f"The query returned {count} record(s). "
                    f"The first result has {first_col} = {first_val}."
                )

            print(f"\n📋 Summary: {summary_text}")

        except Exception as e:
            # Safe rule-based fallback — never crashes the notebook
            col_names = list(df.columns)
            first_row = df.iloc[0].to_dict()
            print(
                f"\n📋 Summary: The query returned {len(df)} record(s) for \"{question}\". "
                f"Columns: {col_names}. First result: {first_row}."
            )

    return df


In [11]:
ask("How many students are there in total?");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : How many students are there in total?
------------------------------------------------------------
SQL       : SELECT COUNT(DISTINCT e.student_id) AS total_students FROM ENROLLMENT e;
Result    : 1 row(s)


,total_students
0,500


Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 1 record(s). The first result has total_students = 500.


# **Task 2 Evaluation: Capabilities & Shortcomings**

In [12]:
ask("Show me all the courses being offered in the computer science department.");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me all the courses being offered in the computer science department.
------------------------------------------------------------
SQL       : SELECT c.course_name FROM COURSE c JOIN DEPARTMENT d ON c.dept_id = d.dept_id WHERE d.dept_name = 'Computer Science';
Result    : 7 row(s)


,course_name
0,Introduction to Programming
1,Data Structures
2,Database Systems
3,Operating Systems
4,Software Engineering
5,Artificial Intelligence
6,Computer Networks


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 7 record(s). The first result has course_name = Introduction to Programming.


In [13]:
ask("What is the average capacity across all sections in the SECTION table?");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : What is the average capacity across all sections in the SECTION table?
------------------------------------------------------------
SQL       : SELECT AVG(s.capacity) AS average_capacity FROM SECTION s;
Result    : 1 row(s)


,average_capacity
0,38.541667


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 1 record(s). The first result has average_capacity = 38.541666666666664.


In [14]:
ask("Show me all the courses in desending order by the number of students enrolled in that course.");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me all the courses in desending order by the number of students enrolled in that course.
------------------------------------------------------------
SQL       : SELECT c.course_name, COUNT(e.student_id) AS number_of_students FROM COURSE c JOIN ENROLLMENT e ON c.course_id = e.course_id GROUP BY c.course_name ORDER BY number_of_students DESC NULLS LAST;
[exec error] Execution failed on sql 'SELECT c.course_name, COUNT(e.student_id) AS number_of_students FROM COURSE c JOIN ENROLLMENT e ON c.course_id = e.course_id GROUP BY c.course_name ORDER BY number_of_students DESC NULLS LAST;': no such column: e.course_id


In [15]:
ask("""Show me all the courses in descending order by the number of students enrolled in that course. 
(Hint: Join COURSE to SECTION, and SECTION to ENROLLMENT)""");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me all the courses in descending order by the number of students enrolled in that course. 
(Hint: Join COURSE to SECTION, and SECTION to ENROLLMENT)
------------------------------------------------------------
SQL       : SELECT c.course_name, COUNT(e.student_id) AS total_students FROM COURSE c JOIN SECTION s ON c.course_id = s.course_id JOIN ENROLLMENT e ON s.section_id = e.section_id GROUP BY c.course_name ORDER BY total_students DESC NULLS LAST;
Result    : 30 row(s)


,course_name,total_students
0,Quantum Physics,96
1,Classical Mechanics,92
2,Broadcast Journalism,86
3,Media Ethics,76
4,Genetics,74
5,Cell Biology,72
6,Construction Materials,57
7,Structural Analysis,56
8,Fluid Mechanics,56
9,Calculus I,39


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 30 record(s). The first result has course_name = Quantum Physics.


In [16]:
ask("List out all the students that live in the city Lahore.");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : List out all the students that live in the city Lahore.
------------------------------------------------------------
SQL       : SELECT s.student_id, s.first_name, s.last_name FROM STUDENT s JOIN ADDRESS a ON s.address_id = a.address_id WHERE a.city LIKE '%Lahore%' LIMIT 5;
Result    : 5 row(s)


,student_id,first_name,last_name
0,STU000006,Sana,Haider
1,STU000010,Talha,Khan
2,STU000020,Usman,Butt
3,STU000034,Farhan,Butt
4,STU000042,Junaid,Ali


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 5 record(s). The first result has student_id = STU000006.


In [17]:
ask("Show me which students were marked Absent, along with the date of their absence.");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me which students were marked Absent, along with the date of their absence.
------------------------------------------------------------
SQL       : SELECT s.first_name, s.last_name, a.date FROM ATTENDANCE a JOIN STUDENT s ON a.enrollment_id = s.student_id WHERE a.status = 'A';
Result    : 0 row(s)


,first_name,last_name,date



📋 Summary: No records matched the query.


In [18]:
ask("""Show me which students were marked status 'A', along with the date of their absence. 
(Hint: Join ATTENDANCE to ENROLLMENT using enrollment_id, and then ENROLLMENT to STUDENT using student_id)""");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me which students were marked status 'A', along with the date of their absence. 
(Hint: Join ATTENDANCE to ENROLLMENT using enrollment_id, and then ENROLLMENT to STUDENT using student_id)
------------------------------------------------------------
SQL       : SELECT s.first_name, s.last_name, a.date, a.status FROM ATTENDANCE a JOIN ENROLLMENT e ON a.enrollment_id = e.enrollment_id JOIN STUDENT s ON e.student_id = s.student_id WHERE a.status = 'A' ORDER BY a.date NULLS LAST;
Result    : 931 row(s)


,first_name,last_name,date,status
0,Amna,Javed,2022-01-01,A
1,Erum,Siddiqui,2022-01-09,A
2,Adnan,Hashmi,2022-01-10,A
3,Ahmad,Siddiqui,2022-01-11,A
4,Rida,Mirza,2022-01-11,A
...,...,...,...,...
926,Danyal,Hashmi,2024-12-28,A
927,Shahid,Chaudhry,2024-12-29,A
928,Ahmad,Ansari,2025-01-03,A
929,Shahid,Chaudhry,2025-01-05,A


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 931 record(s). The first result has first_name = Amna.


In [19]:
ask("Find faculty members that are over the age of 50 and teach in the computer science department");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Find faculty members that are over the age of 50 and teach in the computer science department
------------------------------------------------------------
SQL       : SELECT f.first_name, f.last_name FROM FACULTY f JOIN DEPARTMENT d ON f.dept_id = d.dept_id WHERE EXTRACT(YEAR FROM age(now(), f.dob)) > 50 AND d.dept_name = 'Computer Science';
[exec error] Execution failed on sql 'SELECT f.first_name, f.last_name FROM FACULTY f JOIN DEPARTMENT d ON f.dept_id = d.dept_id WHERE EXTRACT(YEAR FROM age(now(), f.dob)) > 50 AND d.dept_name = 'Computer Science';': near "FROM": syntax error


In [20]:
ask("""Find faculty members that are over the age of 50 and teach in the computer science department.
(Hint: In SQLite, calculate age using: (strftime('%Y', 'now') - strftime('%Y', dob)))""");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Find faculty members that are over the age of 50 and teach in the computer science department.
(Hint: In SQLite, calculate age using: (strftime('%Y', 'now') - strftime('%Y', dob)))
------------------------------------------------------------
SQL       : SELECT f.first_name, f.last_name FROM FACULTY f JOIN DEPARTMENT d ON f.dept_id = d.dept_id WHERE (strftime('%Y', 'now') - strftime('%Y', f.dob)) > 50 AND d.dept_name = 'Computer Science';
Result    : 3 row(s)


,first_name,last_name
0,Yasir,Hashmi
1,Amir,Baig
2,Hajra,Qureshi


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 3 record(s). The first result has first_name = Yasir.


In [21]:
ask("""Show the faculty ID, first name, last name, and department name for each faculty member. 
Include the total number of unique sections they teach and the total number of students enrolled in their sections. 
Sort the results by the total number of students in ascending order.""");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show the faculty ID, first name, last name, and department name for each faculty member. 
Include the total number of unique sections they teach and the total number of students enrolled in their sections. 
Sort the results by the total number of students in ascending order.
------------------------------------------------------------
SQL       : SELECT f.faculty_id, f.first_name, f.last_name, d.dept_name, COUNT(DISTINCT s.section_id) AS total_sections, COUNT(DISTINCT e.student_id) AS total_students FROM FACULTY f JOIN DEPARTMENT d ON f.dept_id = d.dept_id JOIN SECTION s ON f.faculty_id = s.faculty_id JOIN ENROLLMENT e ON s.section_id = e.section_id GROUP BY f.faculty_id, f.first_name, f.last_name, d.dept_name ORDER BY total_students ASC;
Result    : 42 row(s)


,faculty_id,first_name,last_name,dept_name,total_sections,total_students
0,23,Fiza,Bhatti,Mathematics,1,11
1,6,Hajra,Qureshi,Computer Science,4,12
2,15,Usman,Ahmed,Business Administration,2,12
3,8,Omar,Iqbal,Electrical Engineering,2,13
4,11,Hina,Javed,Electrical Engineering,2,13
5,39,Rizwan,Rana,Biotechnology,1,13
6,24,Vaneeza,Ahmed,Mathematics,2,15
7,4,Kamran,Ahmed,Computer Science,3,16
8,17,Lubna,Lodhi,Business Administration,2,16
9,31,Imran,Hashmi,Civil Engineering,1,17


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 42 record(s). The first result has faculty_id = 23.


In [22]:
ask("Show me the students that paid their fees using Online Payment method ")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me the students that paid their fees using Online Payment method 
------------------------------------------------------------
SQL       : SELECT s.first_name, s.last_name, p.payment_date, p.payment_method, p.payment_status FROM STUDENT s JOIN PAYMENT p ON s.student_id = p.student_id WHERE p.payment_method = 'Online Payment';
[exec error] Execution failed on sql 'SELECT s.first_name, s.last_name, p.payment_date, p.payment_method, p.payment_status FROM STUDENT s JOIN PAYMENT p ON s.student_id = p.student_id WHERE p.payment_method = 'Online Payment';': no such table: PAYMENT


In [23]:
ask("Show me the first and last names of students who paid their fees using the 'Online' payment method. (Hint: use fee_payment and student tables)");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me the first and last names of students who paid their fees using the 'Online' payment method. (Hint: use fee_payment and student tables)
------------------------------------------------------------
SQL       : SELECT s.first_name, s.last_name FROM fee_payment f JOIN student s ON f.student_id = s.student_id WHERE f.payment_method = 'Online';
Result    : 69 row(s)


,first_name,last_name
0,Ahmad,Siddiqui
1,Amir,Javed
2,Farhan,Qureshi
3,Bisma,Lodhi
4,Danyal,Raza
...,...,...
64,Fiza,Abbasi
65,Luqman,Rana
66,Yasir,Nawaz
67,Waqas,Hussain


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 69 record(s). The first result has first_name = Ahmad.


In [24]:
ask("which department offers the most courses?");


Question : which department offers the most courses?
------------------------------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SQL       : SELECT d.dept_name, COUNT(c.course_id) AS num_courses FROM DEPARTMENT d JOIN COURSE c ON d.dept_id = c.dept_id GROUP BY d.dept_name ORDER BY num_courses DESC LIMIT 1;
Result    : 1 row(s)


,dept_name,num_courses
0,Computer Science,7


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 1 record(s). The first result has dept_name = Computer Science.


In [25]:
ask("Calculate each student's attendance percentage and classify them as able to sit for the final exam or not based on the 75% attendance policy.");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Calculate each student's attendance percentage and classify them as able to sit for the final exam or not based on the 75% attendance policy.
------------------------------------------------------------
SQL       : SELECT s.student_id, s.first_name, s.last_name, CAST(COUNT(a.attendance_id) AS FLOAT) / NULLIF(COUNT(e.enrollment_id), 0) AS attendance_ratio, CASE WHEN CAST(COUNT(a.attendance_id) AS FLOAT) / NULLIF(COUNT(e.enrollment_id), 0) >= 0.75 THEN 'Yes' ELSE 'No' END AS can_sit_for_exam FROM STUDENT s JOIN ENROLLMENT e ON s.student_id = e.student_id JOIN ATTENDANCE a ON e.enrollment_id = a.enrollment_id GROUP BY s.student_id, s.first_name, s.last_name ORDER BY attendance_ratio DESC NULLS LAST;
Result    : 500 row(s)


,student_id,first_name,last_name,attendance_ratio,can_sit_for_exam
0,STU000001,Anila,Haider,1.0,Yes
1,STU000002,Hassan,Ahmed,1.0,Yes
2,STU000003,Tariq,Malik,1.0,Yes
3,STU000004,Ahmad,Siddiqui,1.0,Yes
4,STU000005,Adnan,Hashmi,1.0,Yes
...,...,...,...,...,...
495,STU000496,Khalid,Haider,1.0,Yes
496,STU000497,Imran,Iqbal,1.0,Yes
497,STU000498,Layla,Haider,1.0,Yes
498,STU000499,Asma,Nawaz,1.0,Yes


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 500 record(s). The first result has student_id = STU000001.


In [26]:
ask("Find faculty members who dont live in the city Lahore.");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Find faculty members who dont live in the city Lahore.
------------------------------------------------------------
SQL       : SELECT f.first_name, f.last_name FROM FACULTY f JOIN ADDRESS a ON f.address_id = a.address_id WHERE a.city!= 'Lahore' LIMIT 5;
Result    : 5 row(s)


,first_name,last_name
0,Yasir,Hashmi
1,Amir,Baig
2,Hamza,Raza
3,Kamran,Ahmed
4,Iqra,Raza


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 5 record(s). The first result has first_name = Yasir.


In [27]:
ask("Show me a few students who have low attendance. ");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me a few students who have low attendance. 
------------------------------------------------------------
SQL       : SELECT s.first_name, s.last_name, COUNT(a.status) AS total_absences FROM STUDENT s JOIN ENROLLMENT e ON s.student_id = e.student_id JOIN ATTENDANCE a ON e.enrollment_id = a.enrollment_id WHERE a.status IN ('A', 'L') GROUP BY s.first_name, s.last_name HAVING COUNT(a.status) > 5 ORDER BY total_absences DESC LIMIT 5;
Result    : 5 row(s)


,first_name,last_name,total_absences
0,Junaid,Ansari,12
1,Adnan,Hussain,11
2,Fatima,Sheikh,11
3,Rania,Chaudhry,9
4,Zubair,Hashmi,9


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 5 record(s). The first result has first_name = Junaid.


In [28]:
ask("Show me all students who are not enrolled in any section");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me all students who are not enrolled in any section
------------------------------------------------------------
SQL       : SELECT s.student_id, s.first_name, s.last_name FROM STUDENT s LEFT JOIN ENROLLMENT e ON s.student_id = e.student_id WHERE e.student_id IS NULL;
Result    : 0 row(s)


,student_id,first_name,last_name



📋 Summary: No records matched the query.


In [29]:
ask("Show me all students who are on academic probation.");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me all students who are on academic probation.
------------------------------------------------------------
SQL       : SELECT s.first_name, s.last_name FROM STUDENT s WHERE s.academic_status = 'Academic Probation';
[exec error] Execution failed on sql 'SELECT s.first_name, s.last_name FROM STUDENT s WHERE s.academic_status = 'Academic Probation';': no such column: s.academic_status


In [30]:
ask("""Show me all students who are on academic probation. Hint: A student is on probation if their final_grade in any enrollment is 'F'. Use the ENROLLMENT table. 
""");

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question : Show me all students who are on academic probation. Hint: A student is on probation if their final_grade in any enrollment is 'F'. Use the ENROLLMENT table. 

------------------------------------------------------------
SQL       : SELECT s.student_id, s.first_name, s.last_name FROM STUDENT s JOIN ENROLLMENT e ON s.student_id = e.student_id WHERE e.final_grade = 'F' ORDER BY s.student_id NULLS LAST;
Result    : 46 row(s)


,student_id,first_name,last_name
0,STU000004,Ahmad,Siddiqui
1,STU000014,Lubna,Hussain
2,STU000016,Nadia,Nawaz
3,STU000021,Rida,Gondal
4,STU000035,Usman,Dogar
5,STU000061,Rida,Qureshi
6,STU000066,Waleed,Dogar
7,STU000082,Tahir,Cheema
8,STU000093,Hamza,Javed
9,STU000103,Khalid,Sheikh


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



📋 Summary: The query returned 46 record(s). The first result has student_id = STU000004.


In [31]:
ask("Show me all faculty members who have not been assigned as HOD of any department.");


Question : Show me all faculty members who have not been assigned as HOD of any department.
------------------------------------------------------------
[Out-of-scope] The following term(s) were not found in the database schema and cannot be queried: ['HOD']
Available tables: ['ADDRESS', 'DEPARTMENT', 'FACULTY', 'DEGREE', 'SEMESTER', 'COURSE', 'SECTION', 'STUDENT', 'ENROLLMENT', 'ATTENDANCE', 'FEES', 'FEE_PAYMENT']


# **Bonus A - Ollama Integration**

In [32]:
!sudo apt-get install -y zstd


!curl -fsSL https://ollama.ai/install.sh | sh

print("Ollama installed successfully")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (14.1 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.1

In [33]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(10)
print("Ollama server is running and ready")

Ollama server is running and ready


# **DuckDB Model Setup**

In [34]:
!ollama pull duckdb-nsql
print("Model downloaded successfully")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 08a9f55588ea:   0% ▕                  ▏  30 KB/3.8 GB                  pulling manifest 
pulling 08a9f55588ea:   1% ▕                  ▏  33 MB/3.8 GB                  pulling manifest 
pulling 08a9f55588ea:   3% ▕                  ▏ 119 MB/3.8 GB                  pulling manifest 
pulling 08a9f55588ea:   6% ▕█                 ▏ 217 MB/3.8 GB                  pulling manifest 
pulling 08a9f55588ea:   7% ▕█                 ▏ 260 MB/3.8 GB                  pulling manifest 
pulling 08a9f55588ea:   9% ▕█                 ▏ 360 MB/3.8 GB                  pulling manifest 
pulling 08a9f55588ea:  12% ▕██                ▏ 445 MB/3.8 GB                  pulling manifest 
pulling 08a9f55588ea:  13% ▕██                ▏ 494 MB/3.8 GB                  pulling manifest 
pulling 08a9f55588ea:  15% ▕██                ▏ 587 MB/3.8 GB                  pulling manifest 
pulling 08a9f55588ea:  18

In [35]:


import requests

test_schema = """
CREATE TABLE STUDENT (
    student_id CHAR(9),
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    batch INT,
    dept_id INT
);

CREATE TABLE DEPARTMENT (
    dept_id INT,
    dept_name VARCHAR(100)
);
"""

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "duckdb-nsql",
        "system": f"Here is the database schema that the SQL query will run on: {test_schema}",
        "prompt": "Show me all students in batch 2024",
        "stream": False
    }
)

print("Model response:")
print(response.json()["response"])

Model response:
 SELECT * FROM STUDENT WHERE batch = 2024;


In [36]:
!pip install -q langchain-ollama
print("langchain-ollama installed successfully")

langchain-ollama installed successfully


In [37]:

OLLAMA_SCHEMA = """
DROP DATABASE IF EXISTS dbs_project_prt1;
CREATE DATABASE dbs_project_prt1;
USE dbs_project_prt1;

CREATE TABLE ADDRESS (
    address_id INT NOT NULL AUTO_INCREMENT,
    house_no VARCHAR(10),
    street_lane VARCHAR(50),
    sector_block_phase VARCHAR(50),
    area_town_mohalla VARCHAR(50) NOT NULL,
    city VARCHAR(50) NOT NULL,
    province VARCHAR(50) NOT NULL,
    country VARCHAR(50) NOT NULL DEFAULT 'Pakistan',
    postal_code VARCHAR(12) NOT NULL,
    PRIMARY KEY (address_id)
);

CREATE TABLE DEPARTMENT (
    dept_id INT NOT NULL AUTO_INCREMENT,
    dept_name VARCHAR(100) NOT NULL,
    PRIMARY KEY (dept_id),
    UNIQUE (dept_name)
);

CREATE TABLE FACULTY (
    faculty_id INT NOT NULL AUTO_INCREMENT,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    job_title VARCHAR(50) NOT NULL,
    dob DATE NOT NULL,
    gender ENUM('M', 'F', 'O') NOT NULL,
    address_id INT NOT NULL,
    phone CHAR(15) NOT NULL,
    email VARCHAR(100) NOT NULL CHECK (email LIKE '%@%.%'),
    dept_id INT NOT NULL,
    PRIMARY KEY (faculty_id),
    UNIQUE (phone),
    UNIQUE (email),
    FOREIGN KEY (dept_id) REFERENCES DEPARTMENT (dept_id),
    FOREIGN KEY (address_id) REFERENCES ADDRESS (address_id)
);

ALTER TABLE DEPARTMENT
ADD hod_id INT,
ADD CONSTRAINT fk FOREIGN KEY (hod_id) REFERENCES FACULTY (faculty_id) ON DELETE SET NULL;

CREATE TABLE DEGREE (
    degree_id INT NOT NULL AUTO_INCREMENT,
    degree_title VARCHAR(100) NOT NULL,
    duration TINYINT NOT NULL CHECK (duration BETWEEN 1 AND 6),
    dept_id INT NOT NULL,
    PRIMARY KEY (degree_id),
    UNIQUE (degree_title),
    FOREIGN KEY (dept_id) REFERENCES DEPARTMENT (dept_id)
);

CREATE TABLE SEMESTER (
    sem_id INT NOT NULL AUTO_INCREMENT,
    sem_session VARCHAR(15) NOT NULL,
    start_date DATE NOT NULL,
    end_date DATE NOT NULL,
    sem_status ENUM('Active', 'Completed', 'Upcoming') NOT NULL DEFAULT 'Upcoming',
    PRIMARY KEY (sem_id),
    UNIQUE (sem_session)
);

CREATE TABLE COURSE (
    course_id INT NOT NULL AUTO_INCREMENT,
    course_name VARCHAR(100) NOT NULL,
    credit_hours TINYINT NOT NULL CHECK (credit_hours BETWEEN 1 AND 5),
    dept_id INT NOT NULL,
    PRIMARY KEY (course_id),
    FOREIGN KEY (dept_id) REFERENCES DEPARTMENT (dept_id),
    UNIQUE (course_name)
);

CREATE TABLE SECTION (
    section_id INT NOT NULL AUTO_INCREMENT,
    section_name CHAR(1) NOT NULL,
    capacity INT NOT NULL DEFAULT 30 CHECK (capacity BETWEEN 1 AND 100),
    faculty_id INT NOT NULL,
    course_id INT NOT NULL,
    sem_id INT NOT NULL,
    PRIMARY KEY (section_id),
    FOREIGN KEY (faculty_id) REFERENCES FACULTY (faculty_id),
    FOREIGN KEY (course_id) REFERENCES COURSE (course_id),
    FOREIGN KEY (sem_id) REFERENCES SEMESTER (sem_id)
);

CREATE TABLE STUDENT (
    student_id CHAR(9) NOT NULL,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    batch INT NOT NULL CHECK (batch BETWEEN 2000 AND 2100),
    dob DATE NOT NULL,
    gender ENUM('M', 'F', 'O') NOT NULL,
    address_id INT NOT NULL,
    phone VARCHAR(15) NOT NULL,
    email VARCHAR(100) NOT NULL CHECK (email LIKE '%@%.%'),
    degree_id INT NOT NULL,
    PRIMARY KEY (student_id),
    UNIQUE (phone),
    UNIQUE (email),
    FOREIGN KEY (degree_id) REFERENCES DEGREE (degree_id),
    FOREIGN KEY (address_id) REFERENCES ADDRESS (address_id)
);

CREATE TABLE ENROLLMENT (
    enrollment_id INT NOT NULL AUTO_INCREMENT,
    enrollment_date DATE NOT NULL DEFAULT(CURDATE()),
    final_grade ENUM('A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-', 'D', 'F', 'W') DEFAULT NULL,
    student_id CHAR(9) NOT NULL,
    section_id INT NOT NULL,
    PRIMARY KEY (enrollment_id),
    FOREIGN KEY (student_id) REFERENCES STUDENT (student_id) ON DELETE CASCADE,
    FOREIGN KEY (section_id) REFERENCES SECTION (section_id)
);

CREATE TABLE ATTENDANCE (
    attendance_id INT NOT NULL AUTO_INCREMENT,
    date DATE NOT NULL,
    status ENUM('P', 'A', 'L') NOT NULL DEFAULT 'A',
    enrollment_id INT NOT NULL,
    PRIMARY KEY (attendance_id),
    FOREIGN KEY (enrollment_id) REFERENCES ENROLLMENT (enrollment_id) ON DELETE CASCADE
);

CREATE TABLE FEES (
    fee_id INT NOT NULL AUTO_INCREMENT,
    amount FLOAT NOT NULL CHECK (amount >= 0),
    per_credit_rate FLOAT NOT NULL DEFAULT 0 CHECK (per_credit_rate >= 0),
    enrollment_id INT NOT NULL,
    PRIMARY KEY (fee_id),
    FOREIGN KEY (enrollment_id) REFERENCES ENROLLMENT (enrollment_id) ON DELETE CASCADE
);

CREATE TABLE FEE_PAYMENT (
    payment_id INT NOT NULL AUTO_INCREMENT,
    total_amount_due FLOAT NOT NULL CHECK (total_amount_due >= 0),
    amount_paid FLOAT NOT NULL DEFAULT 0 CHECK (amount_paid >= 0),
    payment_date DATE,
    payment_method ENUM('Cash', 'Online', 'Bank Transfer', 'Cheque') DEFAULT NULL,
    payment_status ENUM('Paid', 'Partial', 'Pending', 'Overdue') NOT NULL DEFAULT 'Pending',
    student_id CHAR(9) NOT NULL,
    sem_id INT NOT NULL,
    PRIMARY KEY (payment_id),
    FOREIGN KEY (student_id) REFERENCES STUDENT (student_id),
    FOREIGN KEY (sem_id) REFERENCES SEMESTER (sem_id)
);

DELIMITER //
CREATE TRIGGER before_fee_insert
BEFORE INSERT ON FEES
FOR EACH ROW
BEGIN
    DECLARE ch INT;
    SELECT c.credit_hours INTO ch
    FROM ENROLLMENT e
    JOIN SECTION s ON e.section_id = s.section_id
    JOIN COURSE c ON s.course_id = c.course_id
    WHERE e.enrollment_id = NEW.enrollment_id;
    
    SET NEW.amount = NEW.per_credit_rate * COALESCE(ch, 0);
END //

CREATE TRIGGER before_fee_update
BEFORE UPDATE ON FEES
FOR EACH ROW
BEGIN
    DECLARE ch INT;
    SELECT c.credit_hours INTO ch
    FROM ENROLLMENT e
    JOIN SECTION s ON e.section_id = s.section_id
    JOIN COURSE c ON s.course_id = c.course_id
    WHERE e.enrollment_id = NEW.enrollment_id;
    
    SET NEW.amount = NEW.per_credit_rate * COALESCE(ch, 0);
END //
DELIMITER ;
"""

print("Schema context built successfully")
print(f"Schema length: {len(OLLAMA_SCHEMA)} characters")

Schema context built successfully
Schema length: 5829 characters


In [38]:
from langchain_ollama import OllamaLLM

ollama_llm = OllamaLLM(
    model="duckdb-nsql",
    temperature=0,
    system=f"Here is the database schema that the SQL query will run on: {OLLAMA_SCHEMA}"
)

print("duckdb-nsql connected to LangChain successfully")

duckdb-nsql connected to LangChain successfully


In [39]:
conn = sqlite3.connect(DB_PATH)
print("Database connection reopened successfully")


test = conn.execute("SELECT COUNT(*) FROM STUDENT").fetchone()
print(f"Student count: {test[0]}")

Database connection reopened successfully
Student count: 500


In [ ]:
import re
import duckdb

def ask_ollama(question):
    print(f"\nQuestion: {question}")
    print("-" * 60)

    try:
        prompt = f"""Here is the database schema:

{OLLAMA_SCHEMA}
You are an SQL expert.
Rules you must follow:
1. Never use GROUP BY ALL. Always list columns explicitly in GROUP BY.
2. Always prefix every column with its table alias e.g. c.course_id not course_id.
3. When using COUNT or any aggregate, always include GROUP BY with all non-aggregated columns explicitly listed.
4. Only return the SQL query. No explanation. No markdown. No code blocks. No comments.
5. Always end your SQL query with a semicolon.
6. Never join tables directly if they do not share a column — always follow the correct join path through intermediate tables.
7. Only use columns and tables that exist in the schema above. Never invent column or table names.
8. When joining ATTENDANCE to STUDENT you must go through ENROLLMENT first — ATTENDANCE joins ENROLLMENT on enrollment_id, then ENROLLMENT joins STUDENT on student_id.
9. Always use table aliases when joining multiple tables.
10. The output must start directly with SELECT, WITH, INSERT, UPDATE or DELETE — nothing before it.

Generate a SQL query to answer this question:
{question}

"""

        raw_output = ollama_llm.invoke(prompt)
        print(f"Raw model output:\n{raw_output}")
        print()

        sql_match = re.search(
            r'(SELECT|UPDATE|DELETE|INSERT|WITH)[\s\S]*?;',
            raw_output,
            re.IGNORECASE | re.DOTALL
        )

        if not sql_match:
            print("Could not find SQL in model output")
            return None

        sql = sql_match.group(0).strip()
        print(f"Extracted SQL:\n{sql}")
        print()

        cursor = conn.execute(sql)
        columns = [description[0] for description in cursor.description]
        rows = cursor.fetchall()

        print(f"Results ({len(rows)} rows returned):")
        print()

        col_widths = [len(col) for col in columns]
        for row in rows[:10]:
            for i, val in enumerate(row):
                col_widths[i] = max(col_widths[i], len(str(val)))

        header = " | ".join(col.ljust(col_widths[i]) for i, col in enumerate(columns))
        print(header)
        print("-" * len(header))

        for row in rows[:10]:
            print(" | ".join(str(val).ljust(col_widths[i]) for i, val in enumerate(row)))

        if len(rows) > 10:
            print(f"... and {len(rows) - 10} more rows")

        summary_prompt = f"""
        The question was: {question}
        The results returned {len(rows)} rows with columns: {columns}
        First few results: {rows[:3]}
        Write a brief 1-2 sentence plain English summary of these results.
        Do not include any SQL in your summary.
        """
        summary = ollama_llm.invoke(summary_prompt)
        print(f"\nSummary: {summary}")

        return {
            "question": question,
            "sql": sql,
            "results": rows,
            "columns": columns,
            "row_count": len(rows),
            "summary": summary
        }

    except Exception as e:
        print(f"Error: {str(e)}")
        return None
            
        

In [41]:
ask_ollama("Show me all the courses being offered in the computer science department.");


Question: Show me all the courses being offered in the computer science department.
------------------------------------------------------------
Raw model output:
 SELECT * FROM COURSE WHERE dept_id = (SELECT dept_id FROM DEPARTMENT WHERE dept_name = 'Computer Science');

Extracted SQL:
SELECT * FROM COURSE WHERE dept_id = (SELECT dept_id FROM DEPARTMENT WHERE dept_name = 'Computer Science');

Results (7 rows returned):

course_id | course_name                 | credit_hours | dept_id
----------------------------------------------------------------
1         | Introduction to Programming | 3            | 1      
2         | Data Structures             | 3            | 1      
3         | Database Systems            | 3            | 1      
4         | Operating Systems           | 3            | 1      
5         | Software Engineering        | 3            | 1      
6         | Artificial Intelligence     | 3            | 1      
7         | Computer Networks           | 3           

In [42]:
ask_ollama("What is the average capacity across all sections in the SECTION table?");


Question: What is the average capacity across all sections in the SECTION table?
------------------------------------------------------------
Raw model output:
 SELECT AVG(capacity) FROM SECTION;

Extracted SQL:
SELECT AVG(capacity) FROM SECTION;

Results (1 rows returned):

AVG(capacity)     
------------------
38.541666666666664

Summary:  SELECT AVG(capacity) FROM SECTION;


In [43]:
ask_ollama("Show me all the courses in descending order by the number of students enrolled in that course.");


Question: Show me all the courses in descending order by the number of students enrolled in that course.
------------------------------------------------------------
Raw model output:
 SELECT * FROM COURSE
ORDER BY (SELECT COUNT(*) FROM ENROLLMENT WHERE ENROLLMENT.course_id = COURSE.course_id) DESC;

Extracted SQL:
SELECT * FROM COURSE
ORDER BY (SELECT COUNT(*) FROM ENROLLMENT WHERE ENROLLMENT.course_id = COURSE.course_id) DESC;

Error: no such column: ENROLLMENT.course_id


In [44]:
ask_ollama("Show me all the courses in descending order by the number of students enrolled in that course. (HINT: Join COURSE to SECTION, and SECTION to ENROLLMENT)");


Question: Show me all the courses in descending order by the number of students enrolled in that course. (HINT: Join COURSE to SECTION, and SECTION to ENROLLMENT)
------------------------------------------------------------
Raw model output:
 SELECT * FROM COURSE
JOIN SECTION ON COURSE.course_id = SECTION.course_id
JOIN ENROLLMENT ON SECTION.section_id = ENROLLMENT.section_id
ORDER BY COUNT(*) DESC;

Extracted SQL:
SELECT * FROM COURSE
JOIN SECTION ON COURSE.course_id = SECTION.course_id
JOIN ENROLLMENT ON SECTION.section_id = ENROLLMENT.section_id
ORDER BY COUNT(*) DESC;

Error: misuse of aggregate: COUNT()


In [45]:
ask_ollama(" List out all the students that live in the city Lahore.");


Question:  List out all the students that live in the city Lahore.
------------------------------------------------------------
Raw model output:
 SELECT * FROM STUDENT WHERE address_id IN (SELECT address_id FROM ADDRESS WHERE city = 'Lahore');

Extracted SQL:
SELECT * FROM STUDENT WHERE address_id IN (SELECT address_id FROM ADDRESS WHERE city = 'Lahore');

Results (38 rows returned):

student_id | first_name | last_name | batch | dob        | gender | address_id | phone       | email                               | degree_id
----------------------------------------------------------------------------------------------------------------------------------------------
STU000006  | Sana       | Haider    | 2021  | 2003-03-18 | F      | 54         | 03344722900 | sana6@student.university.edu.pk     | 13       
STU000010  | Talha      | Khan      | 2020  | 2002-04-03 | M      | 58         | 03335276906 | talha10@student.university.edu.pk   | 3        
STU000020  | Usman      | Butt      | 

In [46]:
ask_ollama("Show me which students were marked status 'A', along with the date of their absence. (Hint: Join ATTENDANCE to ENROLLMENT using enrollment_id, and then ENROLLMENT to STUDENT using student_id)");


Question: Show me which students were marked status 'A', along with the date of their absence. (Hint: Join ATTENDANCE to ENROLLMENT using enrollment_id, and then ENROLLMENT to STUDENT using student_id)
------------------------------------------------------------
Raw model output:
 SELECT ABS(DATEDIFF('day', attendance.date, enrollment.enrollment_date)) AS days_absent, enrollment.student_id
FROM ATTENDANCE attendance
JOIN ENROLLMENT enrollment ON attendance.enrollment_id = enrollment.enrollment_id
WHERE attendance.status = 'A'
ORDER BY days_absent DESC;

Extracted SQL:
SELECT ABS(DATEDIFF('day', attendance.date, enrollment.enrollment_date)) AS days_absent, enrollment.student_id
FROM ATTENDANCE attendance
JOIN ENROLLMENT enrollment ON attendance.enrollment_id = enrollment.enrollment_id
WHERE attendance.status = 'A'
ORDER BY days_absent DESC;

Error: no such function: DATEDIFF


In [47]:

!sudo apt-get install -y zstd
!curl -fsSL https://ollama.ai/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                       17.2%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [48]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(10)
print("Ollama server restarted successfully")

Ollama server restarted successfully


# **Mistral Setup**

In [49]:
!ollama pull mistral
print("Mistral model downloaded successfully")

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling f5074b1221da:   0% ▕                  ▏ 501 KB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   1% ▕                  ▏  33 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   2% ▕                  ▏ 107 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   4% ▕                  ▏ 191 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   5% ▕                  ▏ 237 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   7% ▕█                 ▏ 315 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:   8% ▕█                 ▏ 361 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:  10% ▕█                 ▏ 455 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:  13% ▕██                ▏ 551 MB/4.4 GB                  pulling manifest 
pulling f5074b1221da:  15

In [50]:
!pip install -q langchain-ollama
print("langchain-ollama installed successfully")

langchain-ollama installed successfully


In [51]:
from langchain_ollama import OllamaLLM

mistral_llm = OllamaLLM(
    model="mistral",
    temperature=0
)

print("Mistral connected to LangChain successfully")

Mistral connected to LangChain successfully


In [ ]:
import re

def ask_mistral(question):
    print(f"\nQuestion: {question}")
    print("-" * 60)

    unknown = check_out_of_scope(question)
    if unknown:
        print(f"[Out-of-scope] The following term(s) were not found in the "
              f"database schema and cannot be queried: {unknown}\n"
              f"Available tables: {list(SCHEMA_TABLES.keys())}")
        return None

    try:
        prompt = f"""You are a SQL expert. You only write standard SQL that works in SQLite and MySQL.

STRICT RULES:
1. Never use GROUP BY ALL. Always list columns explicitly in GROUP BY.
2. Always prefix every column with its table alias e.g. c.course_id not course_id.
3. When using COUNT or any aggregate, always include GROUP BY with all non-aggregated columns explicitly listed.
4. Only return the SQL query. No explanation. No markdown. No code blocks. No comments.
5. Always end your SQL query with a semicolon.
6. Never join tables directly if they do not share a column — always follow the correct join path through intermediate tables.
7. Only use columns and tables that exist in the schema below. Never invent column or table names.
8. When joining ATTENDANCE to STUDENT you must go through ENROLLMENT first — ATTENDANCE joins ENROLLMENT on enrollment_id, then ENROLLMENT joins STUDENT on student_id.
9. Always use table aliases when joining multiple tables.
10. The output must start directly with SELECT, WITH, INSERT, UPDATE or DELETE — nothing before it.
11. Never use DATEDIFF, DATEADD, NVL, ISNULL or any non-standard date functions.
12. Never use DuckDB specific syntax — only use standard SQL compatible with SQLite and MySQL.

DATABASE SCHEMA:
{OLLAMA_SCHEMA}

QUESTION: {question}

SQL QUERY:"""

        raw_output = mistral_llm.invoke(prompt)
        print(f"Raw model output:\n{raw_output}")
        print()

        sql_match = re.search(
            r'(SELECT|UPDATE|DELETE|INSERT|WITH)[\s\S]*?;',
            raw_output,
            re.IGNORECASE | re.DOTALL
        )

        if not sql_match:
            print("Could not find SQL in model output")
            return None

        sql = sql_match.group(0).strip()
        print(f"Extracted SQL:\n{sql}")
        print()

        cursor = conn.execute(sql)
        columns = [description[0] for description in cursor.description]
        rows = cursor.fetchall()

        print(f"Results ({len(rows)} rows returned):")
        print()

        col_widths = [len(col) for col in columns]
        for row in rows[:10]:
            for i, val in enumerate(row):
                col_widths[i] = max(col_widths[i], len(str(val)))

        header = " | ".join(col.ljust(col_widths[i]) for i, col in enumerate(columns))
        print(header)
        print("-" * len(header))

        for row in rows[:10]:
            print(" | ".join(str(val).ljust(col_widths[i]) for i, val in enumerate(row)))

        if len(rows) > 10:
            print(f"... and {len(rows) - 10} more rows")

        if len(rows) == 0:
            print("\n📋 Summary: No records matched the query.")
            summary = "No records matched the query."
        else:
            summary_prompt = (
                f"The user asked: \"{question}\"\n"
                f"The SQL query returned {len(rows)} row(s) with columns: {columns}.\n"
                f"First few rows: {rows[:5]}\n\n"
                "Write a concise natural-language summary (2-3 sentences) that directly "
                "answers the question using the actual data. Mention specific names, numbers, "
                "or values from the results. Do NOT include any SQL or code."
            )
            summary = mistral_llm.invoke(summary_prompt)
            summary = re.sub(
                r"(SQLQuery:|SELECT|FROM|WHERE)[\s\S]*", "",
                summary, flags=re.IGNORECASE
            ).strip() or summary.strip()
            print(f"\n📋 Summary: {summary}")

        return {
            "question": question,
            "sql": sql,
            "results": rows,
            "columns": columns,
            "row_count": len(rows),
            "summary": summary
        }

    except Exception as e:
        print(f"Error: {str(e)}")
        return None

print("ask_mistral function defined successfully")

ask_mistral function defined successfully


In [53]:
ask_mistral("Show me all the courses being offered in the computer science department.");


Question: Show me all the courses being offered in the computer science department.
------------------------------------------------------------
Raw model output:
 SELECT c.course_name
FROM COURSE c
JOIN DEPARTMENT d ON c.dept_id = d.dept_id
WHERE d.dept_name = 'Computer Science';

Extracted SQL:
SELECT c.course_name
FROM COURSE c
JOIN DEPARTMENT d ON c.dept_id = d.dept_id
WHERE d.dept_name = 'Computer Science';

Results (7 rows returned):

course_name                
---------------------------
Introduction to Programming
Data Structures            
Database Systems           
Operating Systems          
Software Engineering       
Artificial Intelligence    
Computer Networks          

📋 Summary: The Computer Science department is offering seven courses: Introduction to Programming, Data Structures, Database Systems, Operating Systems, and Software Engineering. These are the courses currently available in the department.


In [54]:
ask_mistral(" List out all the students that live in the city Lahore.");


Question:  List out all the students that live in the city Lahore.
------------------------------------------------------------
Raw model output:
 SELECT s.student_id, s.first_name, s.last_name, a.city
FROM STUDENT s
JOIN ADDRESS a ON s.address_id = a.address_id
WHERE a.city = 'Lahore';

Extracted SQL:
SELECT s.student_id, s.first_name, s.last_name, a.city
FROM STUDENT s
JOIN ADDRESS a ON s.address_id = a.address_id
WHERE a.city = 'Lahore';

Results (38 rows returned):

student_id | first_name | last_name | city  
--------------------------------------------
STU000006  | Sana       | Haider    | Lahore
STU000010  | Talha      | Khan      | Lahore
STU000020  | Usman      | Butt      | Lahore
STU000034  | Farhan     | Butt      | Lahore
STU000042  | Junaid     | Ali       | Lahore
STU000044  | Ali        | Abbasi    | Lahore
STU000051  | Ehsan      | Dogar     | Lahore
STU000058  | Vaneeza    | Aslam     | Lahore
STU000071  | Luqman     | Farooq    | Lahore
STU000091  | Amir       | Ali

In [55]:
ask_mistral("Find faculty members that are over the age of 50 and teach in the computer science department.");


Question: Find faculty members that are over the age of 50 and teach in the computer science department.
------------------------------------------------------------
Raw model output:
 SELECT f.first_name, f.last_name, d.dept_name
FROM FACULTY f
JOIN DEPARTMENT d ON f.dept_id = d.dept_id
WHERE f.dob > '1970-01-01' AND d.dept_name = 'Computer Science';

Extracted SQL:
SELECT f.first_name, f.last_name, d.dept_name
FROM FACULTY f
JOIN DEPARTMENT d ON f.dept_id = d.dept_id
WHERE f.dob > '1970-01-01' AND d.dept_name = 'Computer Science';

Results (5 rows returned):

first_name | last_name | dept_name       
-----------------------------------------
Yasir      | Hashmi    | Computer Science
Amir       | Baig      | Computer Science
Hamza      | Raza      | Computer Science
Kamran     | Ahmed     | Computer Science
Iqra       | Raza      | Computer Science

📋 Summary: Five faculty members in the Computer Science department are over the age of 50: Yasir Hashmi, Amir Baig, Hamza Raza, Kamran A

In [56]:
ask_mistral("""Show the faculty ID, first name, last name, and department name for each faculty member. Include the total number of unique sections they teach and the total number of students enrolled in their sections. Sort the results by the total number of students in ascending order. 
""");


Question: Show the faculty ID, first name, last name, and department name for each faculty member. Include the total number of unique sections they teach and the total number of students enrolled in their sections. Sort the results by the total number of students in ascending order. 

------------------------------------------------------------
Raw model output:
 SELECT f.faculty_id, f.first_name, f.last_name, d.dept_name,
COUNT(DISTINCT s.section_id) AS total_sections,
(SELECT COUNT(DISTINCT e.enrollment_id) FROM ENROLLMENT e JOIN ATTENDANCE a ON e.enrollment_id = a.enrollment_id WHERE f.faculty_id = e.faculty_id AND a.status != 'L') AS total_students
FROM FACULTY f
JOIN DEPARTMENT d ON f.dept_id = d.dept_id
LEFT JOIN SECTION s ON f.faculty_id = s.faculty_id
GROUP BY f.faculty_id, f.first_name, f.last_name, d.dept_name
ORDER BY total_students ASC;

Extracted SQL:
SELECT f.faculty_id, f.first_name, f.last_name, d.dept_name,
COUNT(DISTINCT s.section_id) AS total_sections,
(SELECT COUNT

In [57]:
ask_mistral("Show the faculty ID, first name, last name, and department name for each faculty member. Include the total number of unique sections they teach and the total number of students enrolled in their sections. Sort the results by the total number of students in ascending order. Hint: Join FACULTY to DEPARTMENT using dept_id, then JOIN FACULTY to SECTION using staff_id as the join key, then JOIN SECTION to ENROLLMENT using section_id. Do not join FACULTY directly to ENROLLMENT — there is no faculty_id column in ENROLLMENT.");


Question: Show the faculty ID, first name, last name, and department name for each faculty member. Include the total number of unique sections they teach and the total number of students enrolled in their sections. Sort the results by the total number of students in ascending order. Hint: Join FACULTY to DEPARTMENT using dept_id, then JOIN FACULTY to SECTION using staff_id as the join key, then JOIN SECTION to ENROLLMENT using section_id. Do not join FACULTY directly to ENROLLMENT — there is no faculty_id column in ENROLLMENT.
------------------------------------------------------------
[Out-of-scope] The following term(s) were not found in the database schema and cannot be queried: ['JOIN']
Available tables: ['ADDRESS', 'DEPARTMENT', 'FACULTY', 'DEGREE', 'SEMESTER', 'COURSE', 'SECTION', 'STUDENT', 'ENROLLMENT', 'ATTENDANCE', 'FEES', 'FEE_PAYMENT']


In [58]:
ask_mistral("""Show the faculty ID, first name, last name, and department name for each faculty member. Include the total number of unique sections they teach and the total number of students enrolled in their sections. Sort the results by the total number of students in ascending order. Use only simple JOINs — do not use subqueries. Join FACULTY to DEPARTMENT using dept_id. Join FACULTY to SECTION using f.faculty_id = s.faculty_id. Join SECTION to ENROLLMENT using s.section_id = e.section_id. Group by faculty_id, first_name, last_name and dept_name. Use COUNT(DISTINCT s.section_id) for total sections and COUNT(DISTINCT e.enrollment_id) for total students.""");


Question: Show the faculty ID, first name, last name, and department name for each faculty member. Include the total number of unique sections they teach and the total number of students enrolled in their sections. Sort the results by the total number of students in ascending order. Use only simple JOINs — do not use subqueries. Join FACULTY to DEPARTMENT using dept_id. Join FACULTY to SECTION using f.faculty_id = s.faculty_id. Join SECTION to ENROLLMENT using s.section_id = e.section_id. Group by faculty_id, first_name, last_name and dept_name. Use COUNT(DISTINCT s.section_id) for total sections and COUNT(DISTINCT e.enrollment_id) for total students.
------------------------------------------------------------
[Out-of-scope] The following term(s) were not found in the database schema and cannot be queried: ['DISTINCT', 'COUNT']
Available tables: ['ADDRESS', 'DEPARTMENT', 'FACULTY', 'DEGREE', 'SEMESTER', 'COURSE', 'SECTION', 'STUDENT', 'ENROLLMENT', 'ATTENDANCE', 'FEES', 'FEE_PAYMENT'

In [59]:
ask_mistral("""Show me all students who are on academic probation. Hint: A student is on probation if their final_grade in any enrollment is 'F'. Use the ENROLLMENT table.""");


Question: Show me all students who are on academic probation. Hint: A student is on probation if their final_grade in any enrollment is 'F'. Use the ENROLLMENT table.
------------------------------------------------------------
Raw model output:
 SELECT s.student_id, s.first_name, s.last_name
FROM STUDENT s
JOIN ENROLLMENT e ON s.student_id = e.student_id
WHERE e.final_grade IN ('F')
GROUP BY s.student_id, s.first_name, s.last_name;

Extracted SQL:
SELECT s.student_id, s.first_name, s.last_name
FROM STUDENT s
JOIN ENROLLMENT e ON s.student_id = e.student_id
WHERE e.final_grade IN ('F')
GROUP BY s.student_id, s.first_name, s.last_name;

Results (45 rows returned):

student_id | first_name | last_name
-----------------------------------
STU000004  | Ahmad      | Siddiqui 
STU000014  | Lubna      | Hussain  
STU000016  | Nadia      | Nawaz    
STU000021  | Rida       | Gondal   
STU000035  | Usman      | Dogar    
STU000061  | Rida       | Qureshi  
STU000066  | Waleed     | Dogar    
ST

# **Bonus C: Interactive Chat Interface**

In [61]:

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import traceback, pandas as pd

_CSS = """
<style>
  .nlsql-card{font-family:'Segoe UI',system-ui,sans-serif;background:#0f172a;
    border-radius:12px;padding:24px 28px;margin:8px 0 16px 0;
    box-shadow:0 4px 24px rgba(0,0,0,.45);color:#e2e8f0}
  .nlsql-title{font-size:1.25rem;font-weight:700;letter-spacing:.03em;
    color:#7dd3fc;margin-bottom:4px}
  .nlsql-subtitle{font-size:.78rem;color:#64748b;margin-bottom:20px}
  .nlsql-label{font-size:.72rem;font-weight:600;text-transform:uppercase;
    letter-spacing:.08em;color:#94a3b8;margin-bottom:6px;margin-top:14px}
  .nlsql-sql{background:#1e293b;border-left:3px solid #38bdf8;border-radius:6px;
    padding:12px 16px;font-family:Consolas,'Fira Code',monospace;font-size:.85rem;
    color:#bae6fd;white-space:pre-wrap;word-break:break-word}
  .nlsql-summary{background:#1e293b;border-left:3px solid #34d399;border-radius:6px;
    padding:12px 16px;font-size:.9rem;color:#a7f3d0}
  .nlsql-error{background:#450a0a;border-left:3px solid #ef4444;border-radius:6px;
    padding:12px 16px;font-size:.88rem;color:#fca5a5}
  .nlsql-tbl{border-collapse:collapse;width:100%;font-size:.82rem}
  .nlsql-tbl th{background:#1e3a5f;color:#ffffff;padding:7px 12px;
    text-align:left;font-weight:700;white-space:nowrap}
  .nlsql-tbl td{padding:6px 12px;border-bottom:1px solid #1e293b;
    color:#f1f5f9;white-space:nowrap;font-weight:500}
  .nlsql-tbl tbody tr:nth-child(odd)  td{background:#131c2e}
  .nlsql-tbl tbody tr:nth-child(even) td{background:#182035}
  .nlsql-tbl tr:hover td{background:#1e3a5f;color:#ffffff}
  .nlsql-note{font-size:.75rem;color:#475569;margin-top:6px}
</style>
"""

def _df_to_html(df, max_rows=50):
    p = df.head(max_rows)
    ths = ''.join(f'<th>{c}</th>' for c in p.columns)
    trs = ''.join(
        '<tr>' + ''.join(f'<td>{v}</td>' for v in r) + '</tr>'
        for _, r in p.iterrows()
    )
    note = (f"<p class='nlsql-note'>Showing {max_rows} of {len(df)} rows</p>"
            if len(df) > max_rows else '')
    return (f"<div style='overflow-x:auto'>"
            f"<table class='nlsql-tbl'><thead><tr>{ths}</tr></thead>"
            f"<tbody>{trs}</tbody></table></div>{note}")

BACKEND_MAP = {
    '🤖  SQLcoder   (HuggingFace)' : ask,
    '🦆  DuckDB-NSQL (Ollama)'     : ask_ollama,
    '⚡  Mistral     (Ollama)'     : ask_mistral,
}

_dropdown = widgets.Dropdown(
    options=list(BACKEND_MAP.keys()),
    value=list(BACKEND_MAP.keys())[2],
    description='Model:',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '55px'},
)

_q = widgets.Textarea(
    placeholder='e.g.  Show me all students in Lahore',
    layout=widgets.Layout(width='100%', height='72px'),
)

_run = widgets.Button(
    description='Run',
    icon='database',
    layout=widgets.Layout(width='110px', margin='8px 0 0 0'),
    style={'button_color': '#0ea5e9', 'font_weight': 'bold'},
)

_clr = widgets.Button(
    description='Clear',
    icon='times',
    layout=widgets.Layout(width='100px', margin='8px 0 0 8px'),
    style={'button_color': '#334155'},
)

_lbl = widgets.Label(value='')
_out = widgets.Output()

def _on_run(_):
    question = _q.value.strip()
    if not question:
        _lbl.value = '⚠️  Please type a question first.'
        return

    selected_label   = _dropdown.value
    selected_backend = BACKEND_MAP[selected_label]
    model_name       = selected_label.split('(')[0].strip().replace('🤖','').replace('🦆','').replace('⚡','').strip()

    _lbl.value = f'⏳  Running on {model_name} — please wait…'
    _run.disabled = True

    with _out:
        clear_output(wait=True)
        try:
            result = selected_backend(question)

            if result is None:
                display(HTML(_CSS +
                    "<div class='nlsql-card'><div class='nlsql-error'>"
                    "⚠️  No result returned.<br>"
                    "Possible reasons:<br>"
                    "&nbsp;&nbsp;• The question contains a term not in the database schema<br>"
                    "&nbsp;&nbsp;• The model could not generate valid SQL<br>"
                    "&nbsp;&nbsp;• The generated SQL failed to execute<br><br>"
                    "Check the output printed above this card for details."
                    "</div></div>"))
                _lbl.value = '⚠️  No result returned.'
                _run.disabled = False
                return

            sql_h  = str(result.get('sql', '—')).replace('<','&lt;').replace('>','&gt;')
            summ_h = str(result.get('summary', '')).replace('<','&lt;')
            rows   = result.get('results', [])
            cols   = result.get('columns', [])
            n      = result.get('row_count', 0)

            if cols and rows is not None:
                tbl_h = _df_to_html(pd.DataFrame(rows, columns=cols))
            else:
                tbl_h = "<p style='color:#64748b;margin:8px 0'>No tabular data returned.</p>"

            display(HTML(_CSS + f"""
              <div class='nlsql-card'>
                <div class='nlsql-title'>🔍 Query Result</div>
                <div class='nlsql-subtitle'>
                  Model: {model_name} &nbsp;·&nbsp; {n} row(s) returned
                </div>

                <div class='nlsql-label'>Generated SQL</div>
                <div class='nlsql-sql'>{sql_h}</div>

                <div class='nlsql-label'>Results</div>
                {tbl_h}

                <div class='nlsql-label'>Summary</div>
                <div class='nlsql-summary'>{summ_h or '—'}</div>
              </div>
            """))
            _lbl.value = f'✅  Done — {n} row(s) returned using {model_name}.'

        except Exception as exc:
            safe = str(exc).replace('<','&lt;').replace('>','&gt;')
            display(HTML(_CSS +
                f"<div class='nlsql-card'><div class='nlsql-error'>"
                f"<strong>❌ Unexpected error:</strong><br>{safe}"
                f"</div></div>"))
            _lbl.value = '❌  An error occurred.'
            print(traceback.format_exc())

    _run.disabled = False

def _on_clear(_):
    _q.value   = ''
    _lbl.value = ''
    with _out:
        clear_output()

_run.on_click(_on_run)
_clr.on_click(_on_clear)

_q.observe(
    lambda c: _on_run(None)
    if c['new'].endswith('\n') and not c['new'].endswith('\n\n') else None,
    names='value',
)

display(HTML(_CSS + """
<div class='nlsql-card' style='margin-bottom:0'>
  <div class='nlsql-title'>🗄️ NL → SQL Explorer</div>
  <div class='nlsql-subtitle'>
    University Management Database &nbsp;·&nbsp;
    Select a model, type your question, press Enter or click Run
  </div>
</div>
"""))

display(widgets.VBox([
    _dropdown,
    _q,
    widgets.HBox([_run, _clr]),
    _lbl,
    _out,
]))